# Tutor only · mol-CSPy run of T2 for notebook 04

Makes `data/t2_molcspy/`. About 15 min on a free T4 GPU.

1. Runtime → Change runtime type → T4 GPU, then Runtime → Run all.
2. Results go to `MyDrive/t2_molcspy_run/`. If the session drops, Run all again; finished structures are skipped.
3. The last cell downloads `t2_molcspy.zip`. Unzip it into `hands_on/data/t2_molcspy/`.

In [ ]:
# @title 1 · Settings
REPO = "https://github.com/fmggggg/agentic-ai-for-functional-materials-discovery"   # or put hands_on.zip in My Drive (or /content) and set REPO = None
PLAN = "2:40,14:30,15:10,4:10,19:10"   # space group:count, 100 structures
TIME_LIMIT_MIN = 150
DRIVE_OUT = "/content/drive/MyDrive/t2_molcspy_run"

In [ ]:
# @title 2 · Get the code, mount Drive, install (~2 min)
import os
from google.colab import drive
drive.mount("/content/drive")
if not os.path.exists("/content/hands_on"):
    if REPO:
        !git clone -q {REPO} /content/hands_on
    else:   # hands_on.zip uploaded to /content or to the top of My Drive
        ZIP = "/content/hands_on.zip" if os.path.exists("/content/hands_on.zip") else "/content/drive/MyDrive/hands_on.zip"
        !mkdir -p /content/hands_on && unzip -q {ZIP} -d /content/hands_on
%cd /content/hands_on
!pip -q install mace-torch pymatgen rdkit oxtalign average-minimum-distance toml "requests==2.32.4"
!nvidia-smi --query-gpu=name,memory.total --format=csv
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
# @title 3 · Generate with mol-CSPy (CPU, ~1 min; skipped if already done)
if not os.path.exists(f"{DRIVE_OUT}/generated.csv"):
    !python scripts/run_molcspy_t2.py generate --out {DRIVE_OUT} --plan {PLAN}
else:
    print("already generated")

In [ ]:
# @title 4 · Relax with MACE-OFF (GPU; re-run this cell until it says 0 still to relax)
!python scripts/run_molcspy_t2.py relax --out {DRIVE_OUT} --device cuda --dtype float32 --time-limit {TIME_LIMIT_MIN}

In [ ]:
# @title 5 · Score against the 5 references with OXtalign (CPU, ~5 min)
!python scripts/run_molcspy_t2.py analyse --out {DRIVE_OUT}

In [ ]:
# @title 6 · Download the small result folder
!cd {DRIVE_OUT} && rm -f t2_molcspy.zip && zip -qr t2_molcspy.zip t2_molcspy && ls -lh t2_molcspy.zip
from google.colab import files
files.download(f"{DRIVE_OUT}/t2_molcspy.zip")
drive.flush_and_unmount()   # make sure everything written to Drive is actually synced